# 01 · Exploring the UCI Adult dataset + baseline model

**Goal:** understand the data before modeling, then train a first (baseline) model to beat.

Run each cell with **Shift+Enter**. Make sure the kernel (top right) is set to `venv`.

Sections:
1. Load the data
2. The target: who earns >$50K?
3. Numeric features
4. Categorical features
5. Missing values
6. Baseline model
7. Takeaways

## 1. Load the data
`load_dataset()` (in `deploymate/data/load.py`) downloads nothing; it reads `data/raw/` and fixes the file quirks (see `docs/DATASET.md`).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # so `deploymate` imports from the notebooks/ folder

import pandas as pd
import plotly.express as px

from deploymate.data.load import load_dataset, NUMERIC, CATEGORICAL

X_train, X_test, y_train, y_test = load_dataset()
train = X_train.assign(income=y_train.map({0: "<=50K", 1: ">50K"}))

print("Train:", X_train.shape, " Test:", X_test.shape)
train.head()

## 2. The target: who earns >$50K?
The classes are **imbalanced**: about 3 in 4 people earn <=50K. A model that always says "<=50K" would already be ~76% accurate, which is why we judge models on **F1 and ROC-AUC**, not just accuracy.

In [ ]:
counts = train["income"].value_counts()
print(counts, "\n")
print("Share >50K:", round(y_train.mean(), 3))
px.bar(x=counts.index, y=counts.values, labels={"x": "income", "y": "people"},
       title="Target balance (train)")

## 3. Numeric features
Summary statistics, then how each feature differs between the two income groups.

In [ ]:
train[NUMERIC].describe().round(1)

In [ ]:
px.histogram(train, x="age", color="income", barmode="overlay", nbins=40,
             title="Age by income: higher earners skew older")

In [ ]:
px.box(train, x="income", y="hours_per_week",
       title="Hours per week by income: higher earners work longer hours")

In [ ]:
px.histogram(train, x="education_num", color="income", barmode="group",
             title="Years of education by income: a strong signal")

In [ ]:
zero_gain = (train["capital_gain"] == 0).mean()
print(f"{zero_gain:.1%} of people have capital_gain = 0, a very skewed feature.")
print("Share >50K when capital_gain > 0:", round((train[train.capital_gain > 0].income == ">50K").mean(), 3))

## 4. Categorical features
For each category: how common it is, and what share of that group earns >50K.

In [ ]:
def rate_by(col):
    t = (train.groupby(col)["income"].apply(lambda s: (s == ">50K").mean())
              .rename("share_over_50K").to_frame())
    t["people"] = train[col].value_counts()
    return t.sort_values("share_over_50K", ascending=False).round(3)

rate_by("relationship")

In [ ]:
t = rate_by("occupation").reset_index()
px.bar(t, x="occupation", y="share_over_50K", hover_data=["people"],
       title="Share earning >50K by occupation")

In [ ]:
rate_by("marital_status")

In [ ]:
top = train["native_country"].value_counts(normalize=True).head(5).round(3)
print("native_country is dominated by one value:\n", top)

## 5. Missing values
The raw files write missing values as `?`. The loader turned them into an `"Unknown"` category instead of dropping rows (live data has gaps too).

In [ ]:
unknown = (train[CATEGORICAL] == "Unknown").sum()
unknown[unknown > 0]

## 6. Baseline model
A **Logistic Regression** inside a Pipeline (`deploymate/model/pipeline.py`):
- numeric columns → `StandardScaler` (same scale for every feature)
- categorical columns → `OneHotEncoder` (one yes/no column per category)

Trained on `adult.data`, scored on the untouched `adult.test`.

In [ ]:
from deploymate.model.pipeline import build_model, score

baseline = build_model().fit(X_train, y_train)

results = pd.DataFrame({"train": score(baseline, X_train, y_train),
                        "test": score(baseline, X_test, y_test)})
results.loc["accuracy", "always_<=50K"] = round(1 - y_test.mean(), 4)
results

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, baseline.predict(X_test))
px.imshow(cm, text_auto=True, x=["pred <=50K", "pred >50K"], y=["true <=50K", "true >50K"],
          color_continuous_scale="Blues", title="Baseline confusion matrix (test)")

In [ ]:
# Which features push predictions toward >50K? (Logistic Regression coefficients)
names = baseline.named_steps["preprocess"].get_feature_names_out()
coefs = pd.Series(baseline.named_steps["classifier"].coef_[0], index=names)
top = pd.concat([coefs.nlargest(8), coefs.nsmallest(8)]).sort_values()
px.bar(x=top.values, y=top.index, orientation="h",
       labels={"x": "coefficient (→ more likely >50K)", "y": ""},
       title="Strongest baseline features", height=500)

## 7. Takeaways
- **Imbalanced target (24% >50K):** report F1 + ROC-AUC, not just accuracy.
- **Strongest signals:** marital status / relationship (married), education, capital gain, age, hours worked.
- **Skewed features:** `capital_gain` is 0 for ~92% of people, and `native_country` is ~90% United-States. Drift in these will be subtle, which is a good stress test for our monitor.
- **Baseline to beat:** see the test column above. Next step: compare tree models (Random Forest, Gradient Boosting) with cross-validation.